### Libraries:

In [ ]:
import numpy as np
import pandas as pd
import ast

#### Data Loading:

In [ ]:
movies = pd.read_csv("Datasets/tmdb_5000_movies.csv")
credits = pd.read_csv("Datasets/tmdb_5000_credits.csv")

In [ ]:
movies.head(1)

In [ ]:
credits.head(1)

In [ ]:
movies.shape

In [ ]:
credits.shape

#### Merging:

In [ ]:
movies = movies.merge(credits,on="title")
movies.head(1)

In [ ]:
movies.columns

#### Selecting Main Features:

In [ ]:
movies = movies [["movie_id","title","overview","genres","keywords","cast","crew"]]
movies.shape

#### Removing Null Values:

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)
movies.shape

#### Checking Duplicate:

In [ ]:
movies.duplicated().sum()

#### Extracting Values:

In [ ]:
def convert(text):
    l=[]
    for i in ast.literal_eval(text):
        l.append(i["name"])
    return l

#### Applying this function

In [ ]:
movies["genres"] = movies["genres"].apply(convert)

In [ ]:
movies["keywords"] = movies["keywords"].apply(convert)

#### Extracting Cast:

In [ ]:
def convert_cast(text):
    l=[]
    counter=0
    for i in ast.literal_eval(text):
        if counter<3:
            l.append(i["name"])
        else:
            break
        counter+=1
    return l

Applying Function to cast column.
Also making cast dataframe for future use

In [ ]:
movies["cast"] = movies["cast"].apply(convert_cast)
cast_df=movies [["movie_id","title","cast"]]
cast_df

In [ ]:
movies.head(1)

#### Function to fetch director from crew data

In [ ]:
def fetch_director(text):
    l=[]
    for i in ast.literal_eval(text):
        if i["job"] == "Director":
            l.append(i["name"])
            break
        
    return l

Applying it to crew column. Making Director for future use

In [ ]:
movies["crew"] = movies["crew"].apply(fetch_director)
director=movies [["movie_id","title","crew"]]

In [ ]:
Genres=movies[["movie_id","title","genres"]]

#### spliting each word and converting into list

In [ ]:
movies["overview"] = movies["overview"].apply(lambda x:x.split())

In [ ]:
movies.head(1)

#### Function to remove space between words

In [ ]:
def remove_space(word):
    l=[]
    for i in word:
        l.append(i.replace(" ",""))
        
    return l

#### Applying it to all lists

In [ ]:
# Removing space because in vectorization each word is treated differently
movies["cast"] = movies["cast"].apply(remove_space)
movies["crew"] = movies["crew"].apply(remove_space)
movies["genres"] = movies["genres"].apply(remove_space)
movies["keywords"] = movies["keywords"].apply(remove_space)

In [ ]:
movies.head(1)

#### Combining all columns into single column

In [ ]:
movies["tags"]=movies["overview"]+movies["genres"]+movies["keywords"]+movies["cast"]+movies["crew"]

In [ ]:
new_df = movies[["movie_id","title","tags"]]
new_df.head(1)

#### Joining all elements of list with space and then changing it to lower casing.

In [ ]:
new_df.loc[:,"tags"] = new_df["tags"].apply(lambda x: " ".join(x)).apply(lambda x:x.lower())
new_df.iloc[0]["tags"]

#### Function to change word into its basic form.

In [ ]:
import nltk
from nltk.stem import PorterStemmer

ps=PorterStemmer()

def stems(text):
    l=[]
    for i in text.split():
        l.append(ps.stem(i))
    
    return " ".join(l)


#### Applying it to tags

In [ ]:
new_df.loc[:,"tags"]=new_df["tags"].apply(stems)

In [ ]:
new_df.iloc[0]["tags"]

#### Implementing CountVectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv=CountVectorizer(max_features=5000,stop_words="english")

In [ ]:
vector=cv.fit_transform(new_df["tags"]).toarray()
vector.shape

In [ ]:
vector[0]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)
similarity.shape

In [ ]:
new_df[new_df["title"] == "Spider-Man"].index[0]

In [ ]:
similarity[0]

#### Function to recommend movie on the given input movie

In [ ]:
def recommend(movie):
    index=new_df[new_df["title"] == movie].index[0]
    distance=sorted(list(enumerate(similarity[index])),reverse=True ,key=lambda x:x[1])
    for i in distance[1:6]:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend("Spider-Man")

#### Function to recommend other movies of this character

In [ ]:
def recommend_with_cast(movie):
    actor=cast_df[cast_df["title"]==movie]["cast"].iloc[0][0]
    same_actor=[]
    for index,row in cast_df.iterrows():
        if actor in row["cast"]:
            print(row["title"])
            same_actor.append(row["title"])
    print(actor)
    return same_actor

In [ ]:
lst=recommend_with_cast("Spider-Man")
import random
random.sample(lst,5)

In [ ]:
def recommend_Dir(movie):
    index = director[director['title'] == movie].index[0]
    L=[]
    for i in director.index:
        if director.loc[i, "crew"] == director.loc[index,"crew"] and director.loc[i, "title"] != director.loc[index,"title"] : 
            print(director.iloc[i].title)
            L.append(director.iloc[i])   
    if len(L)==0:
        L.append(director.iloc[index])
    return L 

In [ ]:
recommend_Dir("Pirates of the Caribbean: At World's End")

In [ ]:
Genres['genres'] = Genres['genres'].apply(lambda x: " ".join(x))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv2 = CountVectorizer(max_features=13,stop_words='english')

In [ ]:
vector2 = cv2.fit_transform(Genres['genres']).toarray()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity2 = cosine_similarity(vector2)

In [ ]:
def recommend_genres(movie):
    index = Genres[new_df['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity2[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:6]:
        print(Genres.iloc[i[0]].title)

In [ ]:
recommend_genres("Drillbit Taylor")

In [ ]:
new_df

In [ ]:
import pickle
pickle.dump(new_df,open("Pickle\movie_list.pkl","wb"))
pickle.dump(similarity,open("Pickle\similarity.pkl","wb"))
pickle.dump(movies,open("Pickle\movies.pkl","wb"))
pickle.dump(director,open("Pickle\director.pkl","wb"))
pickle.dump(cast_df,open("Pickle\Actor.pkl","wb"))
pickle.dump(Genres,open("Pickle\Genres.pkl","wb"))
pickle.dump(similarity2,open("Pickle\similarity2.pkl","wb"))


### Checking Unique director names

In [ ]:
director


In [ ]:
temp=director[:]
temp['crew']=temp['crew'].apply(lambda x: ' '.join(x))

In [ ]:
list=temp['crew'].unique()
counted_values = pd.Series(list).value_counts()

print(counted_values)

In [ ]:
temp[temp['crew']=='Michael Cohn']

In [ ]:
new_df.info()